# Stock Market Prediction System
## Random Forest, XGBoost & SVM with Technical Indicators

**Three classifiers trained on 30+ technical features to predict next-day stock price direction.**

Pipeline:
1. Data Collection & Preprocessing
2. Feature Engineering (30+ indicators)
3. Random Forest — ensemble with calibrated probabilities
4. XGBoost — gradient boosting with strong regularisation
5. SVM — RBF kernel with feature scaling
6. Three-model comparison with ROC curves
7. Walk-forward backtesting (10-fold time-series CV)

## Setup and Imports

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import config
from config import print_config

from src.data_collection import download_stock_data, download_multiple_stocks, load_stock_data
from src.preprocessing import preprocess_stock_data
from src.feature_engineering import engineer_all_features, prepare_ml_data
from src.models import (
    split_train_test,
    train_random_forest, evaluate_model,
    train_svm, evaluate_svm,
    plot_confusion_matrix, plot_feature_importance,
)
from src.xgboost_model import (
    train_xgboost, evaluate_xgboost_model,
    plot_xgb_confusion_matrix, plot_xgb_feature_importance,
)
from src.backtesting import walk_forward_validation, plot_backtest_results
from comparison.compare_models import run_full_comparison
from main import run_complete_pipeline, run_multiple_stocks

print('All imports successful!')

## Configuration

Central settings live in `config.py`. Edit that file to change tickers, date range, or model hyperparameters.

In [ ]:
config.print_config()

## Step 1: Data Pipeline

Download 5 years of OHLCV data from Yahoo Finance, clean it, and engineer 30+ technical indicators.

Indicators include: RSI, MACD (normalised), Bollinger Band width/position, ATR (normalised), OBV change, SMAs, momentum windows, lagged returns, volume ratio, and a price-based sentiment proxy.

In [ ]:
TICKER = 'AAPL'

# Load existing data or download fresh
raw_data = load_stock_data(TICKER, 'raw')
if raw_data is None:
    print(f'Downloading {TICKER} from Yahoo Finance...')
    raw_data = download_stock_data(TICKER, config.START_DATE, config.END_DATE, save=True)

print(f'Raw data shape: {raw_data.shape}')
print(raw_data.tail(3))

# Preprocess: drop unnecessary columns, forward-fill gaps, validate
processed_data = preprocess_stock_data(raw_data)
print(f'\nProcessed data shape: {processed_data.shape}')

# Feature engineering: 30+ technical indicators
feature_data = engineer_all_features(processed_data)
feature_data.to_csv(config.get_data_path(TICKER, 'features'), index=False)

# Prepare ML arrays (drops NaN rows from rolling window warmup)
X, y, feature_names = prepare_ml_data(feature_data)
print(f'\nML dataset: {X.shape[0]} samples x {X.shape[1]} features')
print(f'Target balance (0=down, 1=up): {y.value_counts().to_dict()}')
print(f'\nAll features ({len(feature_names)}):')
for f in feature_names:
    print(f'  {f}')

## Step 2: Chronological Train/Test Split

We split **chronologically** — never randomly — to avoid look-ahead bias.  
The last 20% of rows form the held-out test set; the model never sees future data during training.

In [ ]:
X_train, X_test, y_train, y_test = split_train_test(X, y)

## Step 3: Random Forest

Plain Random Forest without `class_weight='balanced'` — the 53/47 class imbalance is mild enough that balanced weighting only compresses predicted probabilities and hurts threshold reliability.

Key hyperparameters (`config.RF_PARAMS`): `n_estimators=300`, `max_depth=4`, `min_samples_leaf=15`.

In [ ]:
rf_model = train_random_forest(X_train, y_train)
rf_metrics = evaluate_model(rf_model, X_train, y_train, X_test, y_test)

rf_pred = rf_model.predict(X_test)
plot_confusion_matrix(y_test, rf_pred, title='Random Forest -- Test Set')
plot_feature_importance(rf_model, feature_names, top_n=15)

## Step 4: XGBoost

Gradient boosting with strong regularisation to prevent memorisation on ~1 250 training samples.

Key hyperparameters (`config.XGB_PARAMS`): `max_depth=3`, `min_child_weight=10`, `reg_lambda=3.0`, `learning_rate=0.05`.

In [ ]:
xgb_model = train_xgboost(X_train, y_train)
xgb_metrics, xgb_pred = evaluate_xgboost_model(xgb_model, X_train, y_train, X_test, y_test)

plot_xgb_confusion_matrix(y_test, xgb_pred, title='XGBoost -- Test Set')
plot_xgb_feature_importance(xgb_model, feature_names, top_n=15)

## Step 5: Support Vector Machine (SVM)

RBF-kernel SVM with `StandardScaler` feature normalisation. Low regularisation (`C=0.5`) keeps a wide margin — important for noisy financial data. `gamma='scale'` normalises by `n_features * X.var()`.

> SVM can be slow on large datasets. For 5 years of data (~1 250 train samples) it typically finishes in under a minute.

In [ ]:
svm_model, svm_scaler = train_svm(X_train, y_train)
svm_metrics = evaluate_svm(svm_model, svm_scaler, X_train, y_train, X_test, y_test)

svm_pred = svm_model.predict(svm_scaler.transform(X_test))
plot_confusion_matrix(y_test, svm_pred, title='SVM -- Test Set')

## Step 6: Three-Model Comparison

Trains all three models on the same chronological 80/20 split and generates:
- ROC curves (AUC comparison)
- Metrics bar chart (accuracy, precision, recall, F1)
- Feature importance comparison (RF vs XGBoost)

Results are saved to `outputs/comparison_results.csv`.

In [ ]:
comparison_results = run_full_comparison(ticker=TICKER)

## Step 7: Walk-Forward Backtesting

Simulates real deployment: train on all past data, predict the next window, expand the window, repeat across 10 folds (`TimeSeriesSplit`).

Earlier folds have less training data so accuracy is typically lower than the held-out test set. The last 5 folds (most recent market regime) are most informative.

In [ ]:
backtest_results = walk_forward_validation(X, y, feature_names, n_splits=config.N_SPLITS)
plot_backtest_results(backtest_results)

print(f'\nWalk-forward summary (RF, {config.N_SPLITS} folds):')
print(f'  Mean accuracy: {backtest_results["accuracy"].mean():.4f}')
print(f'  Std accuracy:  {backtest_results["accuracy"].std():.4f}')
print(f'  Min / Max:     {backtest_results["accuracy"].min():.4f} / {backtest_results["accuracy"].max():.4f}')

## Results Summary

Side-by-side metrics for all three models on the held-out test set.

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        'Model': 'Random Forest',
        'Test Accuracy': rf_metrics['test_accuracy'],
        'Test Precision': rf_metrics['test_precision'],
        'Test Recall': rf_metrics['test_recall'],
        'Test F1': rf_metrics['test_f1'],
        'Train Accuracy': rf_metrics['train_accuracy'],
    },
    {
        'Model': 'XGBoost',
        'Test Accuracy': xgb_metrics['accuracy'],
        'Test Precision': xgb_metrics['precision'],
        'Test Recall': xgb_metrics['recall'],
        'Test F1': xgb_metrics['f1'],
        'Train Accuracy': xgb_metrics['train_accuracy'],
    },
    {
        'Model': 'SVM',
        'Test Accuracy': svm_metrics['test_accuracy'],
        'Test Precision': svm_metrics['test_precision'],
        'Test Recall': svm_metrics['test_recall'],
        'Test F1': svm_metrics['test_f1'],
        'Train Accuracy': svm_metrics['train_accuracy'],
    },
])

summary = summary.set_index('Model')
print(summary.to_string(float_format='{:.4f}'.format))
print(f'\nWalk-forward mean accuracy (RF): {backtest_results["accuracy"].mean():.4f}')

---
## Optional: Full Single-Stock Pipeline

Runs data collection, preprocessing, feature engineering, RF training, baseline comparison, and backtesting in one call. Useful as a quick end-to-end smoke test.

In [ ]:
# Run the complete pipeline for a single stock
# results = run_complete_pipeline('AAPL')
#
# if results:
#     acc = results['model_results']['metrics']['test_accuracy']
#     print(f'Test Accuracy: {acc:.2%}')

## Optional: Multi-Stock Analysis

Runs the full pipeline for every ticker in `config.TICKERS`. Takes 3-5 minutes per stock.

In [ ]:
# Uncomment to run for all tickers in config.TICKERS
# results_all = run_multiple_stocks()
#
# if results_all:
#     for ticker, r in results_all.items():
#         acc = r['model_results']['metrics']['test_accuracy']
#         bt_acc = r['backtest_results']['accuracy'].mean()
#         print(f'{ticker:6s}  test={acc:.2%}  backtest={bt_acc:.2%}')